Установка библиотек

In [3]:
import torch

print("=" * 60)
print("ПРОВЕРКА ОБОРУДОВАНИЯ")
print("=" * 60)

if torch.cuda.is_available():
    print(f" GPU ДОСТУПЕН: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print(" GPU НЕ ДОСТУПЕН!")
    print("   Выполните: Среда выполнения → Сменить среду выполнения → T4 GPU")

print("=" * 60)

print("\n Установка библиотек...")

# Удаляем старые версии
!pip uninstall -y transformers accelerate bitsandbytes langchain langchain-community -q 2>/dev/null

# Устанавливаем новые
!pip install -q transformers>=4.43.0
!pip install -q accelerate>=0.33.0
!pip install -q bitsandbytes>=0.46.1
!pip install -q langchain==0.2.16 langchain-community==0.2.16
!pip install -q wikipedia prometheus-client beautifulsoup4
# Добавьте в конец ЯЧЕЙКИ 1:
!pip install -q fandom-py

print("\n БИБЛИОТЕКИ УСТАНОВЛЕНЫ")
print("\n" + "=" * 60)
print(" ТЕПЕРЬ ВЫПОЛНИТЕ ПЕРЕЗАПУСК СРЕДЫ ВЫПОЛНЕНИЯ:")
print("   Среда выполнения → Перезапустить сеанс")
print("   Затем запустите следующую ячейку")
print("=" * 60)

# Создаём маркер
with open('/content/setup_complete.txt', 'w') as f:
    f.write('done')

ПРОВЕРКА ОБОРУДОВАНИЯ
 GPU ДОСТУПЕН: Tesla T4
   VRAM: 15.6 GB

 Установка библиотек...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
peft 0.18.1 requires accelerate>=0.21.0, which is not installed.

 БИБЛИОТЕКИ УСТАНОВЛЕНЫ

 ТЕПЕРЬ ВЫПОЛНИТЕ ПЕРЕЗАПУСК СРЕДЫ ВЫПОЛНЕНИЯ:
   Среда выполнения → Перезапустить сеанс
   Затем запустите следующую ячейку


Реализация FANDOM AGENT с fandom-py


In [2]:
import os

if not os.path.exists('/content/setup_complete.txt'):
    print(" Сначала выполните ЯЧЕЙКУ 1 и перезапустите Runtime!")
else:
    print(" Установка подтверждена")

# ======================================================
# ИМПОРТЫ
# ======================================================

import torch
import time
import json
import logging
import warnings
import re
from datetime import datetime
from typing import Dict, Any, List, Optional
from google.colab import drive

# ======================================================
# ПАМЯТЬ АГЕНТА
# ======================================================

from dataclasses import dataclass
from collections import deque

@dataclass
class MemoryEntry:
    query: str
    answer: str
    timestamp: datetime
    sources_count: int
    confidence: float
    latency: float

class AgentMemory:
    def __init__(self, max_size: int = 30):
        self.memories: deque = deque(maxlen=max_size)
        self.max_size = max_size

    def add(self, entry: MemoryEntry):
        self.memories.append(entry)

    def find_similar(self, query: str, threshold: float = 0.5) -> Optional[MemoryEntry]:
        query_words = set(query.lower().split())
        for mem in reversed(self.memories):
            mem_words = set(mem.query.lower().split())
            if query_words and mem_words:
                intersection = query_words & mem_words
                union = query_words | mem_words
                similarity = len(intersection) / len(union) if union else 0
                if similarity >= threshold:
                    return mem
        return None

    def get_history(self, n: int = 10) -> List[Dict]:
        return [
            {
                "query": mem.query,
                "answer": mem.answer[:200] + "..." if len(mem.answer) > 200 else mem.answer,
                "timestamp": mem.timestamp.strftime("%H:%M:%S"),
                "sources": mem.sources_count,
                "confidence": mem.confidence
            }
            for mem in list(self.memories)[-n:]
        ]

    def __len__(self):
        return len(self.memories)

warnings.filterwarnings('ignore')
logging.getLogger("transformers").setLevel(logging.ERROR)

print("\n" + "=" * 60)
print("ЗАГРУЗКА QWEN2.5-1.5B-INSTRUCT")
print("=" * 60)

if not torch.cuda.is_available():
    print(" GPU не найден! Включите T4 GPU")
    print("   Среда выполнения → Сменить среду выполнения → T4 GPU")
else:
    print(f" GPU: {torch.cuda.get_device_name(0)}")

try:
    drive.mount('/content/drive', force_remount=False)
    print(" Google Drive смонтирован")
except:
    print(" Google Drive уже смонтирован")

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
logger = logging.getLogger(__name__)

execution_log = []

def log_step(step_type: str, data: Dict[str, Any]):
    entry = {"timestamp": datetime.now().isoformat(), "step_type": step_type, "data": data}
    execution_log.append(entry)
    logger.info(f"{step_type}: {json.dumps(data, ensure_ascii=False)[:200]}")

# ======================================================
# МЕТРИКИ PROMETHEUS
# ======================================================

from prometheus_client import CollectorRegistry, Counter, Histogram, Gauge, generate_latest

registry = CollectorRegistry()
request_counter = Counter("agent_requests_total", "Total requests", registry=registry)
success_counter = Counter("agent_success_total", "Successful searches", registry=registry)
failure_counter = Counter("agent_failure_total", "Failed searches", registry=registry)
latency_histogram = Histogram("agent_latency_seconds", "Request latency", registry=registry, buckets=[5, 10, 20, 30, 45, 60])
steps_gauge = Gauge("agent_steps_per_request", "Steps per request", registry=registry)
confidence_gauge = Gauge("agent_confidence_score", "Confidence in answer (0-1)", registry=registry)

def get_metrics() -> str:
    return generate_latest(registry).decode('utf-8')

# ======================================================
# ЗАГРУЗКА QWEN2.5
# ======================================================

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

try:
    print(f"\n Загрузка {model_name}...")

    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True
    )
    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=500,
        temperature=0.1,
        do_sample=True,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )

    print("\n Qwen2.5-1.5B успешно загружена")
    if torch.cuda.is_available():
        print(f"   GPU память: {torch.cuda.memory_allocated() / 1024 / 1024:.0f} MB")
except Exception as e:
    print(f" Ошибка загрузки: {e}")
    raise

print("=" * 60)

def llm_generate(prompt: str, max_tokens: int = 250, temperature: float = 0.1) -> str:
    """Генерация с контролем температуры"""
    try:
        messages = [{"role": "user", "content": prompt}]
        formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        response = pipe(formatted_prompt, max_new_tokens=max_tokens, temperature=temperature)
        result = response[0]['generated_text']
        if "<|im_start|>assistant" in result:
            result = result.split("<|im_start|>assistant")[-1]
        if "<|im_end|>" in result:
            result = result.split("<|im_end|>")[0]
        return result.strip() if result.strip() else "Ответ не сгенерирован."
    except Exception as e:
        log_step("LLM_ERROR", {"error": str(e)})
        return ""

# ======================================================
# FANDOM API ЧЕРЕЗ fandom-py
# ======================================================

import fandom

class FandomClient:
    """Клиент для поиска по Fandom Wiki через fandom-py библиотеку"""

    FANDOM_WIKIS = {
        "harry_potter": {
            "domain": "harrypotter",
            "url": "https://harrypotter.fandom.com",
            "keywords": ["гарри поттер", "гриффиндор", "слизерин", "хогвартс", "волан-де-морт",
                        "поттер", "гермиона", "рон", "крестраж", "дамблдор", "волдеморт"]
        },
        "marvel": {
            "domain": "marvel",
            "url": "https://marvel.fandom.com",
            "keywords": ["марвел", "marvel", "танос", "мстители", "железный человек",
                        "капитан америка", "тор", "локи", "thanos", "avengers"]
        },
        "lotr": {
            "domain": "lotr",
            "url": "https://lotr.fandom.com",
            "keywords": ["властелин колец", "средиземье", "фродо", "гэндальф", "саурон",
                        "арагорн", "леголас", "кольцо", "кольцо всевластья"]
        },
        "starwars": {
            "domain": "starwars",
            "url": "https://starwars.fandom.com",
            "keywords": ["звёздные войны", "джедай", "ситх", "вейдер", "йода", "люк скайуокер"]
        },
        "gameofthrones": {
            "domain": "gameofthrones",
            "url": "https://gameofthrones.fandom.com",
            "keywords": ["игра престолов", "джон снег", "дейенерис", "ланнистер"]
        },
        "witcher": {
            "domain": "witcher",
            "url": "https://witcher.fandom.com",
            "keywords": ["ведьмак", "геральт", "йеннифэр", "лютик", "цири"]
        },
        "elderscrolls": {
            "domain": "elderscrolls",
            "url": "https://elderscrolls.fandom.com",
            "keywords": ["свитки", "skyrim", "daedra"]
        },
        "warcraft": {
            "domain": "warcraft",
            "url": "https://warcraft.fandom.com",
            "keywords": ["warcraft", "орки", "артас", "иллидан"]
        },
        "dc": {
            "domain": "dc",
            "url": "https://dc.fandom.com",
            "keywords": ["dc", "бэтмен", "супермен", "джокер"]
        }
    }

    def __init__(self):
        self.current_wiki = None
        self.current_wiki_url = None

    def detect_wikis(self, query: str) -> List[str]:
        """Определение наиболее релевантных Fandom Wiki по запросу"""
        query_lower = query.lower()
        detected = []

        for wiki_name, wiki_info in self.FANDOM_WIKIS.items():
            for keyword in wiki_info["keywords"]:
                if keyword.lower() in query_lower:
                    detected.append(wiki_name)
                    break

        return detected[:2] if detected else ["harry_potter"]

    def set_wiki(self, wiki_name: str):
        """Установка активной wiki для fandom-py"""
        wiki_info = self.FANDOM_WIKIS.get(wiki_name)
        if wiki_info:
            fandom.set_wiki(wiki_info["domain"])
            self.current_wiki = wiki_name
            self.current_wiki_url = wiki_info["url"]
            log_step("FANDOM_SET_WIKI", {"wiki": wiki_name, "domain": wiki_info["domain"]})
            return True
        return False

    def search_page(self, wiki_name: str, query: str) -> Optional[Dict[str, str]]:
        """Поиск страницы в конкретной Fandom Wiki"""
        try:
            if not self.set_wiki(wiki_name):
                return None

            search_results = fandom.search(query, results=1)

            if not search_results:
                return None

            page_title = search_results[0][0]
            page_id = search_results[0][1]

            page = fandom.page(pageid=page_id)

            content = page.content if hasattr(page, 'content') else str(page)
            content = re.sub(r'\s+', ' ', content)

            page_url = f"{self.current_wiki_url}/wiki/{page_title.replace(' ', '_')}"

            return {
                "title": page_title,
                "url": page_url,
                "content": content[:2500],
                "wiki": wiki_name,
                "wiki_url": self.current_wiki_url
            }

        except Exception as e:
            logger.warning(f"Fandom ошибка поиска{wiki_name}: {e}")
            return None

    def search(self, query: str, max_sites: int = 2) -> Dict[str, str]:
        """Поиск по нескольким Fandom Wiki"""
        wikis = self.detect_wikis(query)
        results = {}

        for wiki_name in wikis[:max_sites]:
            result = self.search_page(wiki_name, query)
            if result:
                key = f"[{result['wiki']}] {result['title']}"
                results[key] = f" **Источник: {result['wiki']}**\n🔗 {result['url']}\n\n{result['content']}"
                log_step("FANDOM_HIT", {"wiki": wiki_name, "title": result['title']})
            time.sleep(0.3)

        return results

# ======================================================
# УЛУЧШЕННЫЙ FANDOM EXPERT AGENT (ИСПРАВЛЕННЫЙ)
# ======================================================

class FandomExpertAgent:
    def __init__(self):
        self.steps_count = 0
        self.memory = AgentMemory()
        self.fandom = FandomClient()
        self.detected_fandom = None

    def detect_fandom(self, user_query: str) -> str:
        """Определение вселенной на основе ключевых слов"""
        query_lower = user_query.lower()

        rules = [
            ("lotr", ["фродо", "кольцо", "властелин", "гэндальф", "саурон", "средиземье", "frodo", "gandalf"]),
            ("harry_potter", ["гарри поттер", "гриффиндор", "хогвартс", "волдеморт", "крестраж", "harry potter", "voldemort"]),
            ("marvel", ["марвел", "танос", "мстители", "thanos", "avengers", "iron man"]),
            ("starwars", ["звёздные войны", "джедай", "вейдер", "star wars", "jedi"]),
            ("gameofthrones", ["игра престолов", "джон снег", "дейенерис", "game of thrones"]),
            ("witcher", ["ведьмак", "геральт", "witcher", "geralt"]),
            ("dc", ["dc", "бэтмен", "супермен", "batman", "superman"]),
        ]

        for fandom, keywords in rules:
            for kw in keywords:
                if kw in query_lower:
                    log_step("FANDOM_DETECTED_BY_RULE", {"fandom": fandom, "keyword": kw})
                    return fandom

        detect_prompt = f"""Определи вселенную. Варианты: harry_potter, marvel, lotr, starwars.
Вопрос: {user_query}
Вселенная:"""
        response = llm_generate(detect_prompt, max_tokens=20, temperature=0.1)
        for fandom in ["harry_potter", "marvel", "lotr", "starwars"]:
            if fandom in response.lower():
                return fandom
        return "unknown"

    def plan_search(self, user_query: str) -> List[str]:
        self.steps_count += 1

        self.detected_fandom = self.detect_fandom(user_query)
        log_step("FANDOM_DETECTED", {"fandom": self.detected_fandom})

        plan_prompt = f"""Составь 2 поисковых запроса для поиска в Fandom Wiki по вселенной "{self.detected_fandom if self.detected_fandom != 'unknown' else 'любой'}".

Вопрос: {user_query}

Правила:
- Используй английские названия персонажей/предметов
- Не добавляй нумерацию, пиши каждый запрос с новой строки
- Только запросы, без лишних слов

Запросы:"""

        response = llm_generate(plan_prompt, max_tokens=200, temperature=0.1)

        queries = []
        for line in response.split('\n'):
            line = line.strip()
            if line and len(line) > 3 and not line.startswith(('Запросы', 'Вопрос', 'Правила')):
                line = re.sub(r'^\d+\.\s*', '', line)
                queries.append(line)

        if not queries:
            words = user_query.split()[:3]
            queries = [' '.join(words), user_query[:40]]

        log_step("PLANNING", {"queries": queries[:2], "detected_fandom": self.detected_fandom})
        return queries[:2]

    def search_all(self, queries: List[str]) -> Dict[str, str]:
        self.steps_count += 1

        all_results = {}
        for query in queries:
            results = self.fandom.search(query, max_sites=2)
            all_results.update(results)
            time.sleep(0.3)

        log_step("SEARCH_SUMMARY", {"total_results": len(all_results)})
        return all_results

    def generate_answer(self, user_query: str, search_results: Dict[str, str]) -> str:
        self.steps_count += 1

        if not search_results:
            return f""" ИНФОРМАЦИЯ НЕ НАЙДЕНА В FANDOM

По запросу "{user_query}" ничего не найдено.

💡 Рекомендации:
1. Используйте точное название персонажа/предмета
2. Попробуйте английское название
3. Уточните вселенную"""

        filtered_results = {}
        if self.detected_fandom != "unknown":
            for key, content in search_results.items():
                if self.detected_fandom in key.lower():
                    filtered_results[key] = content
            if not filtered_results:
                filtered_results = search_results
        else:
            filtered_results = search_results

        facts = "\n\n---\n\n".join([
            f"📚 Источник {i+1}:\n{content[:800]}"
            for i, (key, content) in enumerate(list(filtered_results.items())[:2])
        ])

        answer_prompt = f"""Ты - эксперт по вселенным. Отвечай только на основе информации из Fandom Wiki.

ВАЖНЫЕ ПРАВИЛА:
1. НЕ СМЕШИВАЙ РАЗНЫЕ ВСЕЛЕННЫЕ
2. Если информации нет в источниках - скажи об этом честно
3. Не выдумывай факты
4. Всегда указывай конкретных персонажей/события из источников

ПРИМЕРЫ ПРАВИЛЬНЫХ ОТВЕТОВ:

Вопрос: Кто главный злодей в Гарри Поттере?
Источник: Лорд Волдеморт (Том Реддл) - темный волшебник, создавший крестражи...
Ответ: Главным злодеем во вселенной Гарри Поттера является Лорд Волдеморт (Том Реддл). Он создал 7 крестражей и стремился к бессмертию и власти над магическим миром.

Вопрос: Куда шел Фродо?
Источник: Фродо Бэггинс отправился в Мордор, чтобы уничтожить Кольцо Всевластья в Роковой Горе...
Ответ: Фродо шел в Мордор, чтобы уничтожить Кольцо Всевластья в жерле Роковой Горы.

ТЕПЕРЬ ОТВЕТЬ НА ТЕКУЩИЙ ВОПРОС:

Вопрос: {user_query}

Информация из Fandom Wiki:
{facts}

Твой ответ (только на основе фактов, без выдумок, в стиле примеров выше):"""

        final_answer = llm_generate(answer_prompt, max_tokens=500, temperature=0.1)

        lower_answer = final_answer.lower()
        question_lower = user_query.lower()

        if "фродо" in question_lower or "кольцо" in question_lower or "властелин" in question_lower:
            if "гарри поттер" in lower_answer or "волдеморт" in lower_answer or "гриффиндор" in lower_answer:
                final_answer = " Ошибка: Ответ был сформирован с смешением вселенных. " + final_answer

        if final_answer and len(final_answer) > 30:
            sources_text = "\n\n Источники (Fandom Wiki):"
            for key in list(filtered_results.keys())[:2]:
                url_match = re.search(r' (https?://[^\s]+)', search_results[key])
                if url_match:
                    sources_text += f"\n   • {url_match.group(1)}"
            final_answer += sources_text
        else:
            final_answer = f"По запросу '{user_query}' информация найдена в Fandom, но ответ не сформирован."

        return final_answer

    def run(self, user_query: str, use_cache: bool = True) -> Dict[str, Any]:
        """Основной метод для запуска агента"""
        log_step("AGENT_START", {"query": user_query})
        self.steps_count = 0
        self.detected_fandom = None

        if use_cache:
            cached = self.memory.find_similar(user_query)
            if cached and cached.confidence > 0.6:
                log_step("MEMORY_HIT", {"cached_query": cached.query})
                return {
                    "queries": [],
                    "sources_count": cached.sources_count,
                    "answer": f" ИЗ ПАМЯТИ (похожий запрос: '{cached.query}')\n\n{cached.answer}",
                    "steps": 0,
                    "from_cache": True,
                    "confidence": cached.confidence
                }

        queries = self.plan_search(user_query)
        results = self.search_all(queries)
        answer = self.generate_answer(user_query, results)

        confidence = min(1.0, len(results) / 1.5)
        if not results:
            confidence = 0.0

        self.memory.add(MemoryEntry(
            query=user_query,
            answer=answer[:500],
            timestamp=datetime.now(),
            sources_count=len(results),
            confidence=confidence,
            latency=0
        ))

        steps_gauge.set(self.steps_count)

        return {
            "queries": queries,
            "sources_count": len(results),
            "answer": answer,
            "steps": self.steps_count,
            "from_cache": False,
            "confidence": confidence
        }

    def get_memory_stats(self) -> Dict:
        return {"size": len(self.memory), "max_size": self.memory.max_size, "history": self.memory.get_history()}


# ======================================================
# ЗАПУСК
# ======================================================

def run_agent(user_query: str, use_cache: bool = True) -> Dict[str, Any]:
    request_counter.inc()
    start_time = time.time()

    print(f"\n ЗАПРОС: {user_query}\n")
    print(" Агент работает...\n")

    try:
        agent = FandomExpertAgent()
        result = agent.run(user_query, use_cache=use_cache)

        latency = time.time() - start_time
        latency_histogram.observe(latency)

        if not result["from_cache"]:
            success_counter.inc()

        confidence_gauge.set(result["confidence"])

        return {
            "success": True,
            "query": user_query,
            "answer": result["answer"],
            "search_queries": result["queries"],
            "sources_count": result["sources_count"],
            "latency_seconds": round(latency, 2),
            "confidence": result["confidence"],
            "steps": result["steps"],
            "from_cache": result["from_cache"]
        }

    except Exception as e:
        latency = time.time() - start_time
        failure_counter.inc()
        log_step("ERROR", {"error": str(e)})
        return {"success": False, "error": str(e), "latency_seconds": round(latency, 2)}


# ======================================================
# ДЕМОНСТРАЦИЯ
# ======================================================

print("\n" + "=" * 60)
print("FANDOM EXPERT AGENT с Qwen2.5")
print("=" * 60)

test_queries = [
    "Кто главный злодей во вселенной Гарри Поттера?",
    "Кто такой Танос в Марвел?",
    "Куда шел Фродо в Властелине Колец?"
]

for i, query in enumerate(test_queries, 1):
    print("\n" + "=" * 60)
    print(f"ТЕСТ {i}/{len(test_queries)}")
    print("=" * 60)

    result = run_agent(query)

    if result["success"]:
        print("\n ОТВЕТ АГЕНТА (из Fandom):")
        print("-" * 40)
        print(result["answer"])
        print("\n МЕТРИКИ:")
        print(f"    Поисковые запросы: {result['search_queries']}")
        print(f"    Источников: {result['sources_count']}")
        print(f"    Время: {result['latency_seconds']} сек")
        print(f"    Уверенность: {result['confidence']:.0%}")
        print(f"    Из кэша: {'Да' if result.get('from_cache') else 'Нет'}")
    else:
        print(f"\n Ошибка: {result['error']}")

print("\n" + "=" * 60)
print(" ТЕСТИРОВАНИЕ ЗАВЕРШЕНО")
print("=" * 60)

 Установка подтверждена

ЗАГРУЗКА QWEN2.5-1.5B-INSTRUCT
 GPU: Tesla T4
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
 Google Drive смонтирован

 Загрузка Qwen/Qwen2.5-1.5B-Instruct...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


 Qwen2.5-1.5B успешно загружена
   GPU память: 4183 MB

FANDOM EXPERT AGENT с Qwen2.5

ТЕСТ 1/3

 ЗАПРОС: Кто главный злодей во вселенной Гарри Поттера?

 Агент работает...


 ОТВЕТ АГЕНТА (из Fandom):
----------------------------------------
Главным злодеем во вселенной Гарри Поттера является Лорд Волдеморт (Том Реддл). Он создал семь крестражей и стремился к бессмертию и власти над магическим миром.

 Источники (Fandom Wiki):
   • https://harrypotter.fandom.com/wiki/Harry_Potter

 МЕТРИКИ:
    Поисковые запросы: ['Who is the main villain in the Harry Potter universe?', 'What is the primary antagonist of the Harry Potter series?']
    Источников: 1
    Время: 8.89 сек
    Уверенность: 67%
    Из кэша: Нет

ТЕСТ 2/3

 ЗАПРОС: Кто такой Танос в Марвел?

 Агент работает...


 ОТВЕТ АГЕНТА (из Fandom):
----------------------------------------
Танос — это величественный и могущественный магический существ из вселенной Marvel Comics. Он известен своей способностью контролировать все тела в

Интерфейс UI для произвольных запросов

In [3]:
from IPython.display import display, clear_output
import ipywidgets as widgets

class InteractiveFandomAgent:
    """Интерактивный агент с UI для произвольных запросов"""

    def __init__(self):
        # Используем единый экземпляр агента вместо создания нового при каждом запросе
        self.agent = FandomExpertAgent()
        self.history = []
        self.stats = {
            'total': 0,
            'success': 0,
            'times': []
        }

    def create_interface(self):
        """Создание интерактивного интерфейса"""

        # Заголовок
        header = widgets.HTML("""
        <div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 100%);
                    padding: 20px;
                    border-radius: 15px;
                    color: white;
                    text-align: center;
                    margin-bottom: 20px;
                    box-shadow: 0 4px 15px rgba(0,0,0,0.3)">
            <h1>🧙 Fandom Fake Agent</h1>
            <p style="font-size: 16px; opacity: 0.9;">Задайте вопрос о Гарри Поттере, Марвел, Властелине колец и других вселенных</p>
        </div>
        """)

        # Панель примеров
        examples_panel = widgets.HTML("""
        <div style="background: #f0f0f0; padding: 10px; border-radius: 10px; margin-bottom: 15px;">
            <b> Примеры запросов:</b><br>
            <span style="color: #4CAF50; cursor: pointer;" onclick="document.getElementById('query_input').value='Расскажи про крестражи в мире Гарри Поттера';">🔮 Крестражи</span> |
            <span style="color: #2196F3; cursor: pointer;" onclick="document.getElementById('query_input').value='Кто такой Танос в Марвел?';">💜 Танос</span> |
            <span style="color: #9C27B0; cursor: pointer;" onclick="document.getElementById('query_input').value='Что такое Кольцо Всевластья?';">💍 Кольцо Всевластья</span> |
            <span style="color: #FF5722; cursor: pointer;" onclick="document.getElementById('query_input').value='Расскажи про Хогвартс';">🏰 Хогвартс</span>
        </div>
        """)

        # Поле ввода
        query_label = widgets.HTML("<b>📝 Ваш вопрос:</b>")
        self.query_input = widgets.Textarea(
            placeholder='Например: Расскажи про крестражи в мире Гарри Поттера',
            layout=widgets.Layout(width='100%', height='100px'),
            style={'description_width': 'initial'}
        )

        # Кнопки
        self.search_btn = widgets.Button(
            description=' Найти ответ',
            button_style='primary',
            layout=widgets.Layout(width='200px', margin='0 10px 0 0')
        )

        self.clear_btn = widgets.Button(
            description=' Очистить',
            button_style='warning',
            layout=widgets.Layout(width='150px', margin='0 10px 0 0')
        )

        self.history_btn = widgets.Button(
            description=' История',
            button_style='info',
            layout=widgets.Layout(width='150px')
        )

        self.memory_btn = widgets.Button(
            description=' Кэш',
            button_style='success',
            layout=widgets.Layout(width='150px')
        )
        self.memory_btn.on_click(self.show_memory)

        # Обновите buttons HBox:
        buttons = widgets.HBox([self.search_btn, self.clear_btn, self.history_btn, self.memory_btn])

        # Статус бар
        self.status = widgets.HTML(
            value="<span style='color: #666;'><i> Готов к работе. Введите вопрос о любой фэнтези-вселенной...</i></span>"
        )

        # Область вывода результатов
        self.output = widgets.Output()

        # История запросов
        self.history_output = widgets.Output()
        self.history_output.layout = widgets.Layout(width='100%', max_height='300px', overflow_y='auto')
        self.history_output.visible = False

        # Панель метрик
        self.metrics_panel = widgets.HTML("""
        <div style="background: #2c3e50; padding: 10px; border-radius: 10px; color: white; margin-top: 10px;">
            <b> Статистика сессии:</b><br>
            Запросов: <span id="total_requests">0</span> |
            Успешно: <span id="success_count">0</span> |
            Среднее время: <span id="avg_time">0</span> сек
        </div>
        """)

        # Прогресс бар
        self.progress = widgets.IntProgress(
            value=0,
            min=0,
            max=100,
            step=1,
            description='Прогресс:',
            bar_style='info',
            layout=widgets.Layout(width='100%', margin='10px 0'),
            visible=False
        )

        # Статистика
        self.stats = {
            'total': 0,
            'success': 0,
            'times': []
        }

        # Привязываем обработчики
        self.search_btn.on_click(self.on_search)
        self.clear_btn.on_click(self.on_clear)
        self.history_btn.on_click(self.show_history)

        # Собираем интерфейс
        ui = widgets.VBox([
            header,
            examples_panel,
            query_label,
            self.query_input,
            buttons,
            self.status,
            self.progress,
            self.output,
            self.history_output,
            self.metrics_panel
        ], layout=widgets.Layout(width='100%', padding='20px'))

        return ui

    def update_stats_display(self):
        """Обновление отображения статистики"""
        avg_time = sum(self.stats['times']) / len(self.stats['times']) if self.stats['times'] else 0
        self.metrics_panel.value = f"""
        <div style="background: #2c3e50; padding: 10px; border-radius: 10px; color: white; margin-top: 10px;">
            <b> Статистика сессии:</b><br>
            Запросов: {self.stats['total']} |
            Успешно: {self.stats['success']} |
            Среднее время: {avg_time:.1f} сек
        </div>
        """

    def show_memory(self, b):
        """Показать содержимое памяти агента"""
        with self.history_output:
            clear_output()
            self.history_output.visible = True

            mem_stats = self.agent.get_memory_stats()

            print(" ПАМЯТЬ АГЕНТА\n")
            print(f" Размер памяти: {mem_stats['size']}/{mem_stats['max_size']}\n")
            print("---\n")

            if mem_stats['size'] == 0:
                print("Память пуста. Задавайте вопросы, чтобы агент запоминал ответы!")
                return

            print(" Кэшированные ответы:\n")

            for i, mem in enumerate(list(self.agent.memory.memories)[-10:], 1):
                print(f"{i}.  {mem.query[:60]}...")
                print(f"    {mem.answer[:150]}...")
                print(f"    Уверенность: {int(mem.confidence*100)}% |  {mem.sources_count} источников")
                print()

    def on_search(self, b):
        """Обработка поиска - использует единый экземпляр агента"""
        query = self.query_input.value.strip()

        if not query:
            with self.output:
                clear_output()
                print(" Пожалуйста, введите вопрос")
            return

        # Обновляем статистику
        self.stats['total'] += 1
        self.update_stats_display()

        # Блокируем кнопку и показываем прогресс
        self.search_btn.disabled = True
        self.search_btn.description = ' Поиск...'
        self.progress.visible = True
        self.progress.value = 25
        self.status.value = "<span style='color: orange;'> Статус: Анализ запроса...</span>"

        with self.output:
            clear_output(wait=True)
            print(f" Запрос: {query}\n")
            print("---\n")

        start_time = time.time()

        try:
            # Шаг 1: Планирование (если не из кэша)
            self.progress.value = 50
            self.status.value = "<span style='color: orange;'> Статус: Планирование поиска...</span>"
            with self.output:
                print(" Шаг 1/3: Проверка памяти...")

            # Используем единый экземпляр агента
            result = self.agent.run(query, use_cache=True)

            if result["from_cache"]:
                with self.output:
                    print("   Найдено в памяти!\n")
                    print(" Шаг 2/3: Использование кэшированного ответа...\n")
            else:
                with self.output:
                    print(f"    Поисковые запросы: {result['queries']}\n")

                # Шаг 2: Поиск
                self.progress.value = 75
                self.status.value = "<span style='color: orange;'> Статус: Поиск в Fandom...</span>"
                with self.output:
                    print(" Шаг 2/3: Поиск в Fandom Wiki...")
                    print(f"    Найдено источников: {result['sources_count']}\n")

                # Шаг 3 уже выполнен внутри agent.run()
                with self.output:
                    print(" Шаг 3/3: Формирование ответа...\n")

            latency = time.time() - start_time

            # Обновляем статистику
            if not result["from_cache"]:
                self.stats['success'] += 1
                self.stats['times'].append(latency)
            self.update_stats_display()

            # Показываем результат
            self.progress.value = 100
            self.status.value = "<span style='color: green;'>✅ Статус: Готово!</span>"

            with self.output:
                print("=" * 50)
                print(" ОТВЕТ АГЕНТА:\n")
                print(result["answer"])
                print("\n" + "=" * 50)
                print(f"\n Метрики запроса:")
                print(f"    Время: {latency:.2f} сек")
                print(f"    Источников: {result['sources_count']}")
                if not result["from_cache"]:
                    print(f"   🔍 Запросов: {result['queries']}")

                confidence_bar = "█" * int(result['confidence'] * 20) + "░" * (20 - int(result['confidence'] * 20))
                print(f"   Уверенность: {confidence_bar} {int(result['confidence'] * 100)}%")
                print(f"   Из памяти: {'Да' if result['from_cache'] else 'Нет'}")

            # Сохраняем в историю UI
            self.history.append({
                'query': query,
                'timestamp': datetime.now().strftime('%H:%M:%S'),
                'status': 'success',
                'latency': f"{latency:.1f}s",
                'sources': result['sources_count'],
                'from_cache': result['from_cache']
            })

        except Exception as e:
            self.status.value = "<span style='color: red;'> Статус: Ошибка</span>"
            with self.output:
                print(f" **Ошибка:** {str(e)}")
            self.history.append({
                'query': query,
                'timestamp': datetime.now().strftime('%H:%M:%S'),
                'status': 'error',
                'error': str(e)[:100]
            })

        finally:
            # Разблокируем кнопку
            self.search_btn.disabled = False
            self.search_btn.description = ' Найти ответ'
            self.progress.visible = False
            self.progress.value = 0

    def on_clear(self, b):
        """Очистка полей"""
        self.query_input.value = ""
        with self.output:
            clear_output()
        with self.history_output:
            clear_output()
        self.history_output.visible = False
        self.status.value = "<span style='color: #666;'><i> Поля очищены. Введите новый вопрос...</i></span>"

    def show_history(self, b):
        """Показать историю запросов"""
        with self.history_output:
            clear_output()
            self.history_output.visible = True

            if not self.history:
                print(" История пуста. Задайте несколько вопросов.")
                return

            print(" ИСТОРИЯ ЗАПРОСОВ\n")
            print("---")

            for i, item in enumerate(reversed(self.history[-10:]), 1):
                status_icon = "✅" if item['status'] == 'success' else "❌" if item['status'] == 'error' else "⏳"
                print(f"{i}. {status_icon} [{item['timestamp']}] {item['query'][:60]}...")
                if item['status'] == 'success':
                    print(f"      ⏱️ {item.get('latency', 'N/A')} | 📚 {item.get('sources', 0)} источников")
                elif item['status'] == 'error':
                    print(f"      ❌ {item.get('error', 'Unknown error')}")
                print()

# ======================================================
# ЗАПУСК ИНТЕРАКТИВНОГО ИНТЕРФЕЙСА
# ======================================================

print("\n" + "=" * 60)
print("ЗАПУСК ИНТЕРАКТИВНОГО ИНТЕРФЕЙСА")
print("=" * 60)

# Создаем и запускаем интерфейс
interactive_agent = InteractiveFandomAgent()
ui = interactive_agent.create_interface()
display(ui)

print("\n" + "=" * 60)
print("ИНТЕРФЕЙС ЗАПУЩЕН")
print("=" * 60)
print("Теперь вы можете задавать любые вопросы в поле ввода выше!")
print(" Примеры: Гарри Поттер, Марвел, Властелин колец, Звёздные войны")
print("=" * 60)


ЗАПУСК ИНТЕРАКТИВНОГО ИНТЕРФЕЙСА



ИНТЕРФЕЙС ЗАПУЩЕН
Теперь вы можете задавать любые вопросы в поле ввода выше!
 Примеры: Гарри Поттер, Марвел, Властелин колец, Звёздные войны
